# Lab 5 Exercise, part 2: approve file writes and add a tool

This notebook solves the Lab 5 stretch challenge: require approval for file changes and add a new tool.

The lab asks before push notifications and human help. This version also asks before `write_file`, `edit_file`, `create_directory`, and `move_file`. Read-only tools stay open.

The new `@tool` currency converter uses Frankfurter's free v2 rate API. It needs no API key.

Subclass `Sidekick` and rebuild only its worker. Keep the lab's MCP sessions, evaluator, task loop, resume flow, and cleanup.

Use the lab's Windows fix for the MCP error log. It does nothing on macOS and Linux.

In [ ]:
import sys

if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

## Imports

Import `Sidekick`, its prompt, its error middleware, and the middleware used to rebuild the worker.

In [ ]:
import os
from datetime import datetime
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join("..", "..")))

import requests
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
    ModelCallLimitMiddleware,
    PIIMiddleware,
    TodoListMiddleware,
)
from langchain_core.tools import tool

import sidekick as sidekick_module
from sidekick import Sidekick, TolerateToolErrors, WORKER_PROMPT

load_dotenv(override=True)

## A tool of our own: the currency converter

Fetch one current rate from Frankfurter v2, convert the amount, and test the tool before giving it to the agent.

In [ ]:
@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount of money from one currency to another using current exchange rates.
    Use three letter currency codes such as USD, EUR or GBP."""
    base = from_currency.upper()
    quote = to_currency.upper()
    response = requests.get(
        f"https://api.frankfurter.dev/v2/rate/{base}/{quote}",
        timeout=20,
    )
    response.raise_for_status()
    data = response.json()
    converted = round(amount * data["rate"], 2)
    return f"{amount:g} {base} is {converted:g} {quote} as of {data['date']}"

print(convert_currency.invoke({"amount": 100, "from_currency": "USD", "to_currency": "EUR"}))

## The subclass: a Sidekick that asks before writing

Call the lab setup, then rebuild the worker with two changes:

1. Add `convert_currency`.
2. Gate the four filesystem tools that can change files.

Keep the rest of the middleware unchanged.

In [ ]:
class ApprovalSidekick(Sidekick):
    """Add approval for file changes and a currency tool."""

    async def setup(self):
        await super().setup()
        self.tools = self.tools + [convert_currency]
        self.worker = create_agent(
            model="openai:gpt-5.4-mini",
            tools=self.tools,
            system_prompt=f"{WORKER_PROMPT}\nToday is {datetime.now():%A %d %B %Y}.",
            middleware=[
                TolerateToolErrors(),
                TodoListMiddleware(),
                PIIMiddleware("email"),
                PIIMiddleware("credit_card", apply_to_tool_results=True),
                ModelCallLimitMiddleware(run_limit=30),
                HumanInTheLoopMiddleware(
                    interrupt_on={
                        "send_push_notification": True,
                        "request_human_help": True,
                        "write_file": True,
                        "edit_file": True,
                        "create_directory": True,
                        "move_file": True,
                    }
                ),
            ],
            checkpointer=self.memory,
        )

## Setup

Point the sandbox at this contribution before setup. The Sidekick creates it if needed. Print the tools to confirm their names.

In [ ]:
sandbox = os.path.abspath("sandbox")
sidekick_module.SANDBOX = sandbox

sidekick = ApprovalSidekick()
await sidekick.setup()
print(f"Sidekick ready with {len(sidekick.tools)} tools:")
for t in sidekick.tools:
    print(" -", t.name)

## A task that exercises both changes

The task needs search, currency conversion, and a file write. The Sidekick should pause before writing.

In [ ]:
task = """Use your search tool to find the current price in USD of an annual Netflix Standard subscription
in the United States. Convert that yearly cost to EUR with your currency tool. Then write a short markdown
note to price_check.md with the monthly price, the yearly cost in USD and the yearly cost in EUR."""

criteria = """price_check.md is written and contains the monthly USD price, the yearly USD cost and the
yearly EUR cost from a real currency conversion."""

history = await sidekick.run_turn(task, criteria, history=[])
print(history[-1]["content"])

## Approve the file write

`resume()` approves the pending write. Run it again if another write needs approval.

In [ ]:
history = await sidekick.resume(history)
print(history[-1]["content"])

In [ ]:
if sidekick.paused:
    history = await sidekick.resume(history)
for entry in history[-2:]:
    print(f"[{entry['role']}] {entry['content']}\n")

## The evidence

Print the plan and the approved file.

In [ ]:
for todo in sidekick.todos:
    print(f"[{todo['status']}] {todo['content']}")

In [ ]:
print((Path(sandbox) / "price_check.md").read_text(encoding="utf-8"))

## Cleanup

Close the MCP sessions and browser.

In [ ]:
sidekick.cleanup()